# 04. Behavioral Analysis & Pattern Construction

**Project**: AI-Driven Predictive Geofencing and Trajectory Analysis for Proactive Safety Monitoring  
**Stage**: Chunk 4 & 4.1 — Behavioral Pattern Descriptors, Multi-Regime Drift & Synthetic Validation  

This notebook demonstrates and inspects the deterministic, explainable behavioral classification layer developed in [`ml/src/behavior.py`](../src/behavior.py) across all 676,697 non-overlapping 120-second analysis windows from the canonical GeoLife dataset.

> **Important Scope & Methodological Notice**:
> - These descriptors represent geometric/kinematic **movement patterns** (`NORMAL`, `PACING`, `LAPPING`, `RANDOM_DRIFT`, `INSUFFICIENT_EVIDENCE`).
> - They are **NOT** clinical diagnoses and are **NOT** GeoLife transportation labels.
> - Natural observations and controlled synthetic benchmarks are strictly separated.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Resolve paths dynamically
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'ml' / 'notebooks'
PROCESSED_DIR = NOTEBOOK_DIR.parent / 'data' / 'processed'
BEHAVIOR_PARQUET = PROCESSED_DIR / 'behavior_windows.parquet'
REPORT_FILE = PROCESSED_DIR / 'behavior_analysis_report.json'

print(f'Behavior Parquet: {BEHAVIOR_PARQUET} (exists: {BEHAVIOR_PARQUET.exists()})')
print(f'Report File:      {REPORT_FILE} (exists: {REPORT_FILE.exists()})')

## 1. Load Behavioral Analysis Summary Report
Inspect the centralized configuration parameters and dataset processing metadata.

In [ ]:
with open(REPORT_FILE, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(f"Dataset:        {report['dataset_name']}")
print(f"Total Windows:  {report['total_windows']:,}")
print(f"Runtime:        {report['runtime_seconds']:.2f}s")
print('\n=== Centralized Configuration Parameters (BehaviorConfig) ===')
for k, v in report['config'].items():
    print(f"  {k:32s}: {v}")

## 2. Natural GeoLife Behavioral Pattern Distribution
Distribution of deterministic behavioral classes across all 676,697 GeoLife analysis windows.

In [ ]:
dist_df = pd.DataFrame([
    {'Behavioral Pattern': k, 'Count': v, 'Percentage': f"{report['class_percentages'][k]:.2f}%"}
    for k, v in report['class_counts'].items()
])
print(dist_df.to_string(index=False))

## 3. Feature Distributions by Behavioral Category
Inspect medians and interquartile ranges across behavioral categories to verify expected kinematic and geometric separation.

In [ ]:
stats_rows = []
for b_class, metrics in report['grouped_feature_stats'].items():
    if b_class == 'INSUFFICIENT_EVIDENCE':
        continue
    row = {
        'Pattern': b_class,
        'Path (m)': f"{metrics['path_distance_m']['median']:.1f}",
        'Displacement (m)': f"{metrics['straight_line_displacement_m']['median']:.1f}",
        'Closure': f"{metrics['path_closure_ratio']['median']:.2f}",
        'Loop Metric': f"{metrics['loop_metric']['median']:.2f}",
        'Pacing Tend': f"{metrics['pacing_tendency']['median']:.2f}",
        'Heading Var': f"{metrics['heading_variability']['median']:.2f}",
        'Entropy': f"{metrics['entropy_directional']['median']:.2f}",
    }
    stats_rows.append(row)

feature_summary_df = pd.DataFrame(stats_rows)
print(feature_summary_df.to_string(index=False))

## 4. Controlled Synthetic Validation Benchmark
We evaluate the behavioral descriptors against mathematically generated archetypes (straight line, zig-zag, linear pacing corridor, closed circular circuit, 2D Brownian drift, and stationary dwell).

In [ ]:
syn_benchmark = report['synthetic_benchmark']
cr = syn_benchmark['classification_report']
cm = syn_benchmark['confusion_matrix']

print(f"Total Synthetic Benchmark Windows: {syn_benchmark['total_synthetic_windows']}")
print('\n=== Pattern Recovery Metrics ===')
metrics_table = []
for cls_name in ['NORMAL', 'PACING', 'LAPPING', 'RANDOM_DRIFT', 'INSUFFICIENT_EVIDENCE', 'macro avg']:
    metrics_table.append({
        'Class': cls_name,
        'Precision': f"{cr[cls_name]['precision']:.2f}",
        'Recall': f"{cr[cls_name]['recall']:.2f}",
        'F1-Score': f"{cr[cls_name]['f1-score']:.2f}",
        'Support': int(cr[cls_name]['support']),
    })
print(pd.DataFrame(metrics_table).to_string(index=False))

print('\n=== Confusion Matrix (True Rows x Predicted Cols) ===')
cm_df = pd.DataFrame(cm)
print(cm_df)

## 5. Multi-Regime Random Drift Validation
Inspect descriptor responses across 4 distinct random-drift regimes:
1. `strongly_diffusive`: Pure isotropic 2D Brownian random walk
2. `weakly_persistent`: Correlated random walk with directional momentum (defaults to NORMAL)
3. `high_turn`: High-turn frequency random walk with frequent sharp direction changes
4. `spatially_dispersed`: Drift with large step variance leading to broader bounding box expansion

In [ ]:
regimes = report.get('random_drift_regimes', {})
regime_rows = []
for r_name, r_data in regimes.items():
    regime_rows.append({
        'Regime': r_name,
        'Detected / Total': f"{r_data['detected_random_drift']}/{r_data['total_windows']}",
        'Recall': f"{r_data['recall']:.2f}",
        'Median Entropy': f"{r_data['median_entropy']:.2f}",
        'Median Head Var': f"{r_data['median_heading_var']:.2f}",
        'Turn Freq (/min)': f"{r_data['median_turn_freq']:.1f}",
        'Median Displ (m)': f"{r_data['median_displacement_m']:.1f}",
    })
print(pd.DataFrame(regime_rows).to_string(index=False))

## 6. Sample Analysis Windows
Load a sample of classified windows from `behavior_windows.parquet`.

In [ ]:
df_sample = pq.read_table(BEHAVIOR_PARQUET).to_pandas().head(10)
cols = ['window_id', 'behavior_class', 'path_distance_m', 'straight_line_displacement_m', 'path_closure_ratio', 'loop_metric', 'pacing_tendency']
print(df_sample[cols].to_string(index=False))